# Пара 1. Бунин в Google Colab: от текста к частотам и графикам

**Дисциплина:** Анализ художественного текста с помощью ИИ и цифровых технологий
**Материал:** И.А. Бунин — «Кавказ», «Холодная осень», «Чистый понедельник»
**Формат:** совместная лабораторная на ноутбуках (Google Colab)

### Зачем Colab
- не нужно ставить Python на каждый компьютер;
- код и графики у всех одинаковые;
- можно сохранить копию на Google Диск.

### Учёный результат пары
К концу занятия вы умеете: загрузить текст по ссылке, посчитать **относительные** частоты смысловых полей, построить столбчатую диаграмму и Zipf-график, связать цифру с 1–2 цитатами.


## 0. Старт (5 минут)
1. Откройте https://colab.research.google.com под своим Google-аккаунтом.
2. **Файл → Загрузить блокнот** → выберите этот `.ipynb`.
3. Сверху справа нажмите **Подключить**.
4. Каждую ячейку запускайте кнопкой Play или Shift+Enter.

Базовый Colab **бесплатный**. Платный Colab Pro для этой лабораторной **не нужен**.


## 1. Загрузка трёх рассказов из учебного корпуса


In [ ]:
import re
import requests
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://pavelaleks.github.io/samarkand_lectures/demo-corpora/bunin-temnye-allei/txt/"
STORIES = {
    "Кавказ": "kavkaz.txt",
    "Холодная осень": "holodnaya-osen.txt",
    "Чистый понедельник": "chistyj-ponedelnik.txt",
}

texts = {}
for title, fname in STORIES.items():
    url = BASE + fname
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    r.encoding = "utf-8"
    texts[title] = r.text
    print(f"{title}: {len(texts[title])} символов, ~{len(texts[title].split())} слов (грубо)")


## 2. Токенизация и частоты
Считаем «словом» последовательность букв. Это учебное упрощение — на занятии обсудим, что теряем.


In [ ]:
STOP = set('''и в во на не что он она они я ты мы вы его её их как а но да же
бы к ко от до по со о об из у за для или если уже только ещё нет ни
это то так все весь вся всё свой своя свое свои который которая которое
которые был была было были есть быть'''.split())

FIELDS = {
    "любовь": ["любов", "любил", "любила", "любило", "любили", "люблю", "любишь",
               "любит", "любим", "любите", "любят", "поцел", "страст"],
    "память": ["памят", "вспомн", "воспомина", "помни", "помню", "помнил", "помнила"],
    "время": ["время", "года", "год", "лет", "день", "дни", "ночь", "утром", "вечером",
              "осень", "зима", "весна", "лето", "тогда", "теперь", "потом"],
}

def tokenize(text: str):
    return re.findall(r"[А-Яа-яЁёA-Za-z]+", text.lower())

def field_hits(tokens, stems):
    n = 0
    for t in tokens:
        for s in stems:
            if t.startswith(s):
                n += 1
                break
    return n

rows = []
for title, text in texts.items():
    tokens = tokenize(text)
    content = [t for t in tokens if t not in STOP and len(t) > 2]
    N = len(tokens)
    row = {"рассказ": title, "токенов": N, "без_стоп": len(content)}
    for field, stems in FIELDS.items():
        hits = field_hits(tokens, stems)
        row[field] = hits
        row[f"{field}_на_1000"] = round(1000 * hits / N, 2) if N else 0
    rows.append(row)

df = pd.DataFrame(rows)
df


## 3. Столбчатая диаграмма: поля на 1000 слов
Сравниваем только относительные частоты: рассказы разной длины.


In [ ]:
plot_df = df.set_index("рассказ")[[c for c in df.columns if c.endswith("_на_1000")]]
plot_df.columns = [c.replace("_на_1000", "") for c in plot_df.columns]

ax = plot_df.plot(kind="bar", figsize=(9, 4), rot=15)
ax.set_ylabel("на 1000 токенов")
ax.set_title("Смысловые поля в трёх рассказах Бунина")
ax.legend(title="поле")
plt.tight_layout()
plt.show()

print("Паспорт графика: единица = токен; нормализация = на 1000; источник = корпус курса.")


## 4. Zipf для «Чистого понедельника»
Голова сверхчастых слов и длинный хвост — норма языка. Ищем аномалии глазами, не объявляем это автоматически «стилем автора».


In [ ]:
from collections import Counter

tokens = [t for t in tokenize(texts["Чистый понедельник"]) if t not in STOP and len(t) > 2]
freq = Counter(tokens)
ranks = list(range(1, 31))
values = [c for _, c in freq.most_common(30)]

plt.figure(figsize=(8, 4))
plt.plot(ranks, values, marker="o")
plt.xlabel("ранг")
plt.ylabel("частота")
plt.title("Zipf (топ-30): Чистый понедельник, без стоп-слов")
plt.tight_layout()
plt.show()

print("Топ-15:")
for w, c in freq.most_common(15):
    print(f"{c:4d}  {w}")


## 5. Возврат к тексту (10–12 минут, вместе)
1. Какое поле сильнее в каком рассказе — совпадает ли это с читательским ожиданием?
2. Найдите одну цитату (2–3 предложения), которая объясняет пик.
3. Что опасного в списке стемов FIELDS?


In [ ]:
notes = '''
1) ...
2) Цитата: «...»
3) Ограничение метода: ...
'''
print(notes)


## 6. Финал пары
Сохраните блокнот: Файл → Сохранить копию на Диске.
На пару 2: дуга тональности и сравнение пространства.
